# 121 — Workflow, subagente y sistema multiagente

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) *Workflow*: el código fija el orden; el LLM solo rellena pasos.
(b) *Agente + subagentes*: el padre decide y delega; los hijos tienen contexto aislado
pero no autonomía de objetivo (es el patrón del sistema de investigación de Anthropic).
(c) *Multiagente*: cada agente tiene objetivo propio y el flujo emerge de la
negociación; nadie posee el control central.

**Ejercicio 2.** Promedio = 2.3/3 ≈ **0.7667**; mínimo = **0.6** (security). Regla:
security < 0.7 → decisión "mejorar seguridad". Coincide con el laboratorio: el
supervisor reporta `overall = 0.7667` pero decide por el hallazgo mínimo — un promedio
aceptable puede ocultar un aspecto crítico.

**Ejercicio 3.** Multiagente = (5 000 + 2 000) + 3 × (25 000 + 3 000) =
7 000 + 84 000 = **91 000 tokens** → factor ≈ **4.55×** frente a 20 000. Para
justificarlo debe aportar algo que el agente único no puede: paralelismo real
(latencia ÷ 3 en los workers), contextos que no caben en una ventana, o permisos
segregados por rol. Anthropic reporta ~15× en su sistema real: nuestro 4.55× es
conservador porque los workers comparten poco contexto.

**Ejercicio 4.** Ver celda siguiente: el contrato se comprueba con asserts, sin asumir
valores internos más allá de lo documentado.


In [ ]:
result = run_lab("multiagent", seed=121)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 2
scores = {"quality": 0.8, "security": 0.6, "documentation": 0.9}
promedio = round(sum(scores.values()) / len(scores), 4)
minimo = min(scores, key=scores.get)
decision = f"mejorar {minimo}" if scores[minimo] < 0.7 else "aprobar"
print(promedio, minimo, decision)

# Ejercicio 3
tokens_multiagente = (5_000 + 2_000) + 3 * (25_000 + 3_000)
factor = tokens_multiagente / 20_000
print(f"multiagente = {tokens_multiagente} tokens → factor {factor:.2f}x")

# Ejercicio 4
result = run_lab("multiagent", seed=121)
workers = result["result"]["workers"]
assert result["kind"] == "multiagent"
assert len(workers) == 3
assert all({"agent", "score", "finding"} <= set(w) for w in workers)
assert result["result"]["supervisor"]["overall"] == round(
    sum(w["score"] for w in workers) / 3, 4)
show(result["result"]["supervisor"])


## Reflexión

1. La decisión "mejorar seguridad" sale del *mínimo* (0.6), no del promedio (0.7667). ¿Qué política de consolidación usarías si un worker pudiera fallar y devolver score 0 por un error técnico y no por un hallazgo real?
2. Este laboratorio es técnicamente un workflow (el código fija supervisor → workers → consolidación). ¿Qué tendría que cambiar exactamente para que fuera un sistema multiagente según la definición de Anthropic/Wooldridge?
3. Si el valor de la tarea no cubre ~15× el coste en tokens de un agente único, ¿cuál de las tres arquitecturas elegirías para este mismo caso y qué perderías?
